<h1>Первая лаброаторная: многослойный персептрон</h1>

In [9]:
import pandas as pd
import numpy as np

In [10]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [11]:
from tensorflow.keras.datasets import fashion_mnist

# Загружаем данные
(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()

In [12]:
x_train = x_train / 255.0
x_test = x_test / 255.0

In [13]:
x_train = x_train.reshape(-1, 28*28)
x_test = x_test.reshape(-1, 28*28) 

In [14]:
train_filter = np.isin(y_train, [0, 3, 9])
x_train = x_train[train_filter]
y_train = y_train[train_filter]

test_filter = np.isin(y_test, [0, 3, 9])
x_test = x_test[test_filter]
y_test = y_test[test_filter]

In [15]:
mapping = {0: 0, 3: 1, 9: 2}
y_train = np.vectorize(mapping.get)(y_train)
y_test = np.vectorize(mapping.get)(y_test)

In [16]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import SGD

def build_model(hp):
    model = Sequential()

    num_hidden_layers = hp.Int('num_hidden_layers', min_value=1, max_value=4)

    activation = hp.Choice('activation', ['relu', 'tanh', 'sigmoid'])

    # Входной слой
    model.add(Dense(
        units=hp.Int('units_input', min_value=64, max_value=256, step=64),
        activation=activation,
        input_shape=(784,)
    ))

    for i in range(1, num_hidden_layers):
        model.add(Dense(
            units=hp.Int(f'units_{i}', min_value=32, max_value=256, step=32),
            activation=activation
        ))

    # Выходной слой
    model.add(Dense(3, activation='softmax'))

    # Компиляция модели
    model.compile(
        optimizer=SGD(learning_rate=hp.Choice('lr', [0.001, 0.01, 0.1])),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model


In [17]:
import keras_tuner as kt
from tensorflow.keras.callbacks import EarlyStopping

tuner = kt.RandomSearch(
    build_model,                     
    objective='val_loss',             # что минимизируем
    max_trials=10,                    # сколько разных моделей попробовать
    executions_per_trial=1,           # сколько раз каждый запускать
    directory='tuner_dir',            # где сохраняется результат
    project_name='my_mlp_search',     # имя проекта
    overwrite=True                    # чтобы перезапускать заново
)

# Запускаем подбор
tuner.search(
    x_train, y_train,
    epochs=30,
    validation_split=0.25,
    callbacks=[EarlyStopping(monitor='val_loss', patience=3)],
    verbose=1
)


Trial 10 Complete [00h 01m 09s]
val_loss: 0.31823307275772095

Best val_loss So Far: 0.06607099622488022
Total elapsed time: 00h 06m 04s


In [18]:
best_model = tuner.get_best_models(num_models=1)[0]


<h6>Лучшая модель</h6>

In [19]:
best_hyperparams = tuner.get_best_hyperparameters(1)[0]
best_hyperparams.values


{'num_hidden_layers': 2,
 'activation': 'relu',
 'units_input': 192,
 'lr': 0.1,
 'units_1': 96,
 'units_2': 160,
 'units_3': 64}

In [21]:
hps = tuner.get_best_hyperparameters(num_trials=10)


<h6>Все модели</h6>

In [26]:
all_trials = list(tuner.oracle.trials.values())

for i, trial in enumerate(all_trials):
    print(f"\n🔢 Модель #{i + 1}")
    print(f"val_loss: {trial.score}")
    print("Гиперпараметры:")
    for key, value in trial.hyperparameters.values.items():
        print(f"  {key}: {value}")


🔢 Модель #1
val_loss: 0.09916144609451294
Гиперпараметры:
  num_hidden_layers: 1
  activation: sigmoid
  units_input: 128
  lr: 0.1

🔢 Модель #2
val_loss: 0.11542834341526031
Гиперпараметры:
  num_hidden_layers: 3
  activation: relu
  units_input: 128
  lr: 0.001
  units_1: 32
  units_2: 32

🔢 Модель #3
val_loss: 0.0964113250374794
Гиперпараметры:
  num_hidden_layers: 1
  activation: tanh
  units_input: 64
  lr: 0.01
  units_1: 256
  units_2: 32

🔢 Модель #4
val_loss: 0.3669964373111725
Гиперпараметры:
  num_hidden_layers: 2
  activation: sigmoid
  units_input: 128
  lr: 0.001
  units_1: 64
  units_2: 256

🔢 Модель #5
val_loss: 0.07336658239364624
Гиперпараметры:
  num_hidden_layers: 1
  activation: tanh
  units_input: 192
  lr: 0.1
  units_1: 256
  units_2: 128

🔢 Модель #6
val_loss: 0.08181091398000717
Гиперпараметры:
  num_hidden_layers: 4
  activation: relu
  units_input: 192
  lr: 0.1
  units_1: 96
  units_2: 256
  units_3: 32

🔢 Модель #7
val_loss: 0.06976161897182465
Гиперпарам

In [27]:
best_model = tuner.hypermodel.build(best_hyperparams)

In [28]:
best_model.fit(x_train, y_train, epochs=10, batch_size=64, validation_split=0.25)

Epoch 1/10
211/211 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.8974 - loss: 0.2728 - val_accuracy: 0.9316 - val_loss: 0.1767
Epoch 2/10
211/211 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9504 - loss: 0.1313 - val_accuracy: 0.9627 - val_loss: 0.1012
Epoch 3/10
211/211 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9600 - loss: 0.1064 - val_accuracy: 0.9658 - val_loss: 0.0961
Epoch 4/10
211/211 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9587 - loss: 0.1044 - val_accuracy: 0.9664 - val_loss: 0.0898
Epoch 5/10
211/211 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9670 - loss: 0.0948 - val_accuracy: 0.9571 - val_loss: 0.1022
Epoch 6/10
211/211 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9666 - loss: 0.0868 - val_accuracy: 0.9644 - val_loss: 0.0875
Epoch 7/10
211/211 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9729 - loss: 0.0741 - val_accuracy: 0.9618 - val_loss: 0.0987
Epoch 8/10
211/211 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9740 - loss: 0.0725 - val_accuracy: 0.

In [29]:
loss, acc = best_model.evaluate(x_test, y_test)
print(f"🎯 Test accuracy: {acc:.4f}")


94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9719 - loss: 0.0913
🎯 Test accuracy: 0.9733


accuracy: 0.9733 ура

Как можем видеть, модель чутка переобучилась. Акураси после десятой эпохи выше акураси на тестовых данных. Акураси на тестовых данных примерно равна акураси на 7 эпохе

Дополним архитектуру батч нормализацией и дропаутом, посмотрим, улучшит ли это результат

дропаут подберем керастюнером

In [ ]:
model = Sequential()

# Первый скрытый слой
model.add(Dense(192, input_shape=(784,)))
model.add(BatchNormalization())
model.add(Activation('relu'))

# Второй скрытый слой
model.add(Dense(96))
model.add(BatchNormalization())
model.add(Activation('relu'))

# Выходной слой
model.add(Dense(3, activation='softmax'))

In [ ]:
model.compile(
    optimizer=SGD(learning_rate=0.1),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [33]:
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Activation

def build_model(hp):
    model = Sequential()

    dropout_rate = hp.Float('dropout_rate', min_value=0.1, max_value=0.6, step=0.1)

    # Первый скрытый слой
    model.add(Dense(192, input_shape=(784,)))
    model.add(BatchNormalization())
    model.add(Activation('relu'))
    model.add(Dropout(dropout_rate))

    # Второй скрытый слой
    model.add(Dense(96))
    model.add(BatchNormalization())
    model.add(Activation('relu'))
    model.add(Dropout(dropout_rate))

    # Выходной слой
    model.add(Dense(3, activation='softmax'))

    model.compile(
        optimizer=SGD(learning_rate=0.1),  # как в твоей топ-модели
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model


In [34]:
import keras_tuner as kt

tuner = kt.RandomSearch(
    build_model,
    objective='val_loss',
    max_trials=10,
    directory='tuner_dropout_bn',
    project_name='dropout_bn_search',
    overwrite=True
)

tuner.search(
    x_train, y_train,
    epochs=20,
    batch_size=64,
    validation_split=0.25
)


Trial 5 Complete [00h 00m 26s]
val_loss: 0.06884025037288666

Best val_loss So Far: 0.06880777329206467
Total elapsed time: 00h 02m 10s


In [35]:
best_hp = tuner.get_best_hyperparameters(1)[0]
print("🔍 Лучший Dropout:", best_hp.get('dropout_rate'))


🔍 Лучший Dropout: 0.30000000000000004


In [36]:
best_model = tuner.hypermodel.build(best_hp)

In [37]:
best_model.fit(x_train, y_train, epochs=10, batch_size=64, validation_split=0.25)

Epoch 1/10
211/211 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.9054 - loss: 0.2424 - val_accuracy: 0.9451 - val_loss: 0.1337
Epoch 2/10
211/211 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9539 - loss: 0.1213 - val_accuracy: 0.9662 - val_loss: 0.0928
Epoch 3/10
211/211 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9606 - loss: 0.1029 - val_accuracy: 0.9618 - val_loss: 0.0979
Epoch 4/10
211/211 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9653 - loss: 0.0946 - val_accuracy: 0.9687 - val_loss: 0.0843
Epoch 5/10
211/211 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9682 - loss: 0.0843 - val_accuracy: 0.9698 - val_loss: 0.0783
Epoch 6/10
211/211 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9707 - loss: 0.0759 - val_accuracy: 0.9751 - val_loss: 0.0721
Epoch 7/10
211/211 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9733 - loss: 0.0716 - val_accuracy: 0.9729 - val_loss: 0.0730
Epoch 8/10
211/211 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9771 - loss: 0.0648 - val_accuracy: 0.

In [38]:
loss, acc = best_model.evaluate(x_test, y_test)
print(f"🎯 Test accuracy: {acc:.4f}")

94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9741 - loss: 0.0888
🎯 Test accuracy: 0.9747


предыдущая accuracy: 0.9733

как видим, акураси подулучшилась, это хорошо